In [1]:
%load_ext autotime

from ast import literal_eval
import json
from pathlib import Path
import re
import time
from tqdm.auto import tqdm

import pandas as pd

from data_snapshot.utils import load_json
from field_labeling import (
    load_prompt,
    analyze_field_profile,
    process_response,
    make_classification_model,
)

# Inputs

In [2]:
# Inputs
SYSTEM_PROMPT_PATH = "prompts/field_labeling_system.md"
USER_PROMPT_PATH = "prompts/field_labeling_user.md"
MODEL_NAME = "gpt-5.4-mini"
SLEEP_SECONDS = 0.2  # safety throttle
OUTPUT_PATH = f"data/field_labeling_results_v2.jsonl"
PROFILES_PATH = "outputs/3.0-field_profiles.csv"
# ontology_v1.csv is the human-augmented version of ontology_v0.csv
ONTOLOGY_PATH = "outputs/3.1-ontology_v1.csv"

# Load data

In [3]:
system_prompt = load_prompt(SYSTEM_PROMPT_PATH)
user_prompt_template = load_prompt(USER_PROMPT_PATH)
df_ontology = pd.read_csv(ONTOLOGY_PATH)
df_profiles = pd.read_csv(PROFILES_PATH)
canonical_names = tuple(
    df_ontology.loc[df_ontology["status"] == "keep", "canonical_name"]
)
ClassificationModel = make_classification_model(canonical_names)

# Prepare prompts

## Render ontology markdown

In [4]:
ontology_template = """### {CANONICAL_NAME}

Definition:
{DEFINITION}

Examples:
{EXAMPLES}"""

ontologies = {}
for _, row in df_ontology.iterrows():
    example_str = "\n".join(["- " + x.strip() for x in row["sample_fields"].split(",")])

    if row["status"] == "keep":
        replace_dict = {
            "{CANONICAL_NAME}": row["canonical_name"],
            "{DEFINITION}": row["definition"],
            "{EXAMPLES}": example_str,
        }
    
        text = ontology_template + "\n"
        for old, new in replace_dict.items():
            text = text.replace(old, new)

        ontologies[row["canonical_name"]] = text

    elif row["status"] == "merge":
        # Do not create but add examples
        text = ontologies[row["merge_into"]]
        text += example_str + "\n"
        ontologies[row["merge_into"]] = text
        
ontology_md = "\n---\n\n".join(ontologies.values())

# Main pipeline

In [5]:
# Build skip list
to_skip = set()
if Path(OUTPUT_PATH).exists():
    with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
        for line in f:
            to_skip.add(json.loads(line)["metadata_field"])

In [6]:
# Smoke test
# df_profiles = df_profiles.sample(1)
# df_profiles = df_profiles.head(1)

In [7]:
# Initialize dir
Path(OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)

# Main loop
with open(OUTPUT_PATH, "a", encoding="utf-8") as out_f:
    for _, field_row in tqdm(df_profiles.iterrows(), total=len(df_profiles)):
        metadata_field = field_row["metadata_field"]
        
        # Skip if in skip list
        if metadata_field in to_skip:
            print(f"Skipped: {metadata_field}")
            continue

        # Base row — every row has the same keys
        row = {
            "metadata_field": metadata_field,
            "model": MODEL_NAME,
            "raw_output": None,
            "usage": None,
            "cost": None,
            "status": None,
            "incomplete_details": None,
            "parsed_output": None,
            "error": None,
        }

        try:
            # Create user prompt from template
            user_prompt = user_prompt_template
            top_description_values_str = "\n".join(["- " + x.strip() for x in literal_eval(field_row["top_description_values"])])
            top_observed_values_str = "\n".join(["- " + x.strip() for x in literal_eval(field_row["top_observed_values"])])
            replace_dict = {
                "{{ONTOLOGY_MARKDOWN}}": ontology_md,
                "{{metadata_field}}": metadata_field,
                "{{top_description_values}}": top_description_values_str,
                "{{top_observed_values}}": top_observed_values_str,
            }
            for old, new in replace_dict.items():
                user_prompt = user_prompt.replace(old, new)

            # Responses API
            response = analyze_field_profile(
                system_prompt=system_prompt,
                user_prompt=user_prompt,
                model=MODEL_NAME,
                classification_model=ClassificationModel,
            )

            # Parse and process response
            result = process_response(response, model=MODEL_NAME)

            # Log data
            row["raw_output"] = result["raw_output"]
            row["usage"] = result["usage"]
            row["cost"] = result["cost"]
            row["status"] = result["status"]
            row["incomplete_details"] = result["incomplete_details"]
            row["parsed_output"] = result["parsed_output"]
            row["error"] = result["error"]  # None on success, message on parse failure

        except Exception as e:
            row["error"] = str(e)

        # Single write point — always executes
        out_f.write(json.dumps(row, ensure_ascii=False) + "\n")
        out_f.flush()

        time.sleep(SLEEP_SECONDS)

  0%|          | 0/833 [00:00<?, ?it/s]

Skipped: geographic_scope
